In [2]:
# === CELDA UNIFICADA: CONCILIACIÓN EXACTA + MAPA GEOPOLÍTICO (FIG 4.1) + TABLA APÉNDICE C ===
import os
import urllib.request
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from scipy.interpolate import griddata

os.makedirs(f"{BASE}/figuras_tesis", exist_ok=True)

# 1. UNIFICACIÓN ESTRICTA A NIVEL DE INMUEBLE ÚNICO (RECID) -> 1 FILA = 1 EDIFICIO FÍSICO
df_rec_unico = (
    df_reg.sort_values("anio")
    .groupby("RECID", as_index=False)
    .agg(
        X_UTM=("X_UTM", "first"),
        Y_UTM=("Y_UTM", "first"),
        municipio=("municipio", "last"),
        ccts=("cct", lambda s: " / ".join(sorted(s.dropna().unique()))),
        n_ccts=("cct", "nunique"),
        es_publica=("tipo_escuela", lambda s: not s.astype(str).str.upper().str.contains("PRIVAD|PARTICULAR").any()),
        alta_marg=("alta_marg", "max"),
        PM25_mean=("PM25_ALL", "mean"),
        PM25_fund=("PM25_FUNDICION_METAL", "mean"),
        lyc_mean=("lyc_std_500", "mean"),
    )
)

# Umbral Decil 10 definido directamente sobre los 2,392 inmuebles únicos de la ZMVM
p90_recid = np.percentile(df_rec_unico["PM25_mean"], 90)
df_rec_unico["top_10_exp"] = df_rec_unico["PM25_mean"] >= p90_recid
df_rec_unico["hipervulnerable"] = (
    df_rec_unico["es_publica"]
    & (df_rec_unico["alta_marg"] == 1)
    & df_rec_unico["top_10_exp"]
)

df_hiper = df_rec_unico[df_rec_unico["hipervulnerable"]].sort_values("PM25_mean", ascending=False).reset_index(drop=True)
df_otras = df_rec_unico[~df_rec_unico["hipervulnerable"]].copy()

# Imprimir cifras exactas conciliadas para Capítulo 4 y Apéndice C
aporte_fund_hiper = 100.0 * df_hiper["PM25_fund"].sum() / df_hiper["PM25_mean"].sum()
lyc_resto_pub = df_rec_unico[df_rec_unico["es_publica"] & ~df_rec_unico["hipervulnerable"]]["lyc_mean"].mean()
n_multi_turno = (df_hiper["n_ccts"] > 1).sum()
n_aulas = len(df_hiper) * 14
capex_mxn = n_aulas * 6500 / 1e6
opex_mxn = n_aulas * 2200 / 1e6

print("=================================================================")
print("CIFRAS EXACTAS CONCILIADAS (CAPÍTULO 4, FIGURA 4.1 Y APÉNDICE C)")
print("=================================================================")
print(f"• Universo inmuebles ZMVM: {len(df_rec_unico):,}")
print(f"• Umbral Decil 10 (P90 sobre RECID): >= {p90_recid:.4f} μg/m³ ({df_rec_unico['top_10_exp'].sum()} inmuebles)")
print(f"• Inmuebles hipervulnerables ÚNICOS: {len(df_hiper)} inmuebles ({df_hiper['n_ccts'].sum()} CCTs; {n_multi_turno} comparten ambos turnos)")
print(f"• Concentración media hipervulnerables: {df_hiper['PM25_mean'].mean():.4f} μg/m³ (aporte fundición: {aporte_fund_hiper:.1f}%)")
print(f"• Lenguaje medio hipervulnerables vs. resto públicas: {df_hiper['lyc_mean'].mean():.1f} vs. {lyc_resto_pub:.1f} pts ({df_hiper['lyc_mean'].mean() - lyc_resto_pub:.1f} pts)")
print(f"• Presupuesto HEPA ({n_aulas} aulas): CAPEX = ${capex_mxn:.2f}M MXN | OPEX = ${opex_mxn:.2f}M MXN/año")
print("\nDistribución municipal exacta:")
print(df_hiper["municipio"].value_counts())

# 2. CARGAR CAPAS VECTORIALES EN UTM 14N (KM)
GEOJSON_STATES = f"{AERMAP_OUT}/mexico_estados.geojson"
GEOJSON_MUN = f"{AERMAP_OUT}/zmvm_municipios_inegi.geojson"

if not os.path.exists(GEOJSON_STATES):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/angelnmara/geojson/master/mexicoHigh.json", GEOJSON_STATES)

gdf_st = gpd.read_file(GEOJSON_STATES).to_crs("EPSG:32614")
gdf_st["geometry"] = gdf_st["geometry"].scale(xfact=0.001, yfact=0.001, origin=(0, 0))
gdf_cdmx = gdf_st[gdf_st.geometry.centroid.x.between(470, 505) & gdf_st.geometry.centroid.y.between(2120, 2165)].copy()

gdf_mun = None
if os.path.exists(GEOJSON_MUN):
    gdf_mun = gpd.read_file(GEOJSON_MUN).to_crs("EPSG:32614")
    gdf_mun["geometry"] = gdf_mun["geometry"].scale(xfact=0.001, yfact=0.001, origin=(0, 0))

# 3. CARGAR FUENTES PESADAS PRIORITARIAS (FUNDICIÓN + VIDRIO)
df_src = pd.read_csv(f"{BASE}/02-data/processed/aermap_out/catalogo_285_fuentes_aermod_ap42.csv")
df_src["Q_MEAN"] = df_src[["Q_GS_2015", "Q_GS_2017", "Q_GS_2019"]].mean(axis=1)
df_src_zmvm = df_src[df_src["X_UTM"] >= 460000].copy()

mask_pesadas = pd.Series(False, index=df_src_zmvm.index)
for col in df_src_zmvm.select_dtypes(include=["object", "string"]).columns:
    if df_src_zmvm[col].astype(str).str.upper().str.contains("FUNDICION|VIDRIO").any():
        mask_pesadas = df_src_zmvm[col].astype(str).str.upper().str.contains("FUNDICION|VIDRIO")
        break
if not mask_pesadas.any():
    col_hs = [c for c in df_src_zmvm.columns if c.upper() in ["HS", "H_S", "ALTURA"]][0]
    mask_pesadas = df_src_zmvm[col_hs].isin([22.0, 45.0])

df_pesadas = df_src_zmvm[mask_pesadas].copy()
df_ligeras = df_src_zmvm[~mask_pesadas].copy()

# 4. INTERPOLACIÓN ESPACIAL DE PM2.5 (KM UTM)
x_km = df_rec_unico["X_UTM"].values / 1000.0
y_km = df_rec_unico["Y_UTM"].values / 1000.0
z_pm = df_rec_unico["PM25_mean"].values

xi = np.linspace(460, 524, 240)
yi = np.linspace(2112, 2194, 260)
XI, YI = np.meshgrid(xi, yi)
ZI = griddata((x_km, y_km), z_pm, (XI, YI), method="linear")

# 5. CONSTRUIR FIGURA 4.1 CON CONTORNOS GEOPOLÍTICOS Y SIN TRASLAPES
plt.rcParams.update({"font.family": "sans-serif", "font.size": 10})
fig = plt.figure(figsize=(15.4, 7.0), dpi=300)
gs = fig.add_gridspec(1, 2, width_ratios=[1.38, 1.0], wspace=0.42)

# --- PANEL A: MAPA GEOPOLÍTICO + PLUMA + INMUEBLES HIPERVULNERABLES ---
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor("#f8f9fa")

# Capas base geopolíticas (Municipios ZMVM y contorno CDMX)
if gdf_mun is not None:
    gdf_mun.plot(ax=ax1, facecolor="#eef1f5", edgecolor="#95a5a6", linewidth=0.55, zorder=1)
gdf_st.boundary.plot(ax=ax1, color="#5d6d7e", linewidth=1.1, zorder=2)
if len(gdf_cdmx) > 0:
    gdf_cdmx.boundary.plot(ax=ax1, color="#2c3e50", linewidth=1.7, linestyle="-", zorder=3)

# Contorno de concentración industrial y umbral Decil 10
levels = np.linspace(0.015, 0.22, 14)
cf = ax1.contourf(XI, YI, np.clip(ZI, 0, 0.22), levels=levels, cmap="YlOrRd", alpha=0.52, extend="max", zorder=2)
ax1.contour(XI, YI, ZI, levels=[p90_recid], colors="#990000", linewidths=1.3, linestyles="--", zorder=4)

# Puntos de escuelas y fuentes
ax1.scatter(df_otras["X_UTM"]/1000.0, df_otras["Y_UTM"]/1000.0, s=6, c="#7f8c8d", alpha=0.32, label=f"Otras secundarias ZMVM (n = {len(df_otras):,})", zorder=4)
ax1.scatter(df_ligeras["X_UTM"]/1000.0, df_ligeras["Y_UTM"]/1000.0, s=16, marker="^", c="#34495e", alpha=0.55, label=f"Otras fuentes RETC (n = {len(df_ligeras)})", zorder=5)
ax1.scatter(
    df_pesadas["X_UTM"]/1000.0, df_pesadas["Y_UTM"]/1000.0,
    s=50 + 170 * (df_pesadas["Q_MEAN"] / df_pesadas["Q_MEAN"].max()),
    marker="^", c="#1a1a1a", edgecolors="white", linewidths=0.7,
    label=f"Fundiciones y vidrieras (n = {len(df_pesadas)})", zorder=6
)
ax1.scatter(
    df_hiper["X_UTM"]/1000.0, df_hiper["Y_UTM"]/1000.0,
    s=82, marker="o", c="#005a9c", edgecolors="white", linewidths=1.1,
    label=f"Inmuebles hipervulnerables (n = {len(df_hiper)})", zorder=7
)

# Etiquetas automáticas apuntando al centroide real de los inmuebles en los 4 municipios principales
posiciones_caja = {
    "NAUCALPAN": (461.5, 2141.0),
    "TLALNEPANTLA": (461.5, 2162.5),
    "ATIZAP": (461.5, 2175.5),
    "ECATEPEC": (506.0, 2175.0),
}
for clave_mun, (x_txt, y_txt) in posiciones_caja.items():
    sub_m = df_hiper[df_hiper["municipio"].str.upper().str.contains(clave_mun)]
    if len(sub_m) > 0:
        cx, cy = (sub_m["X_UTM"]/1000.0).mean(), (sub_m["Y_UTM"]/1000.0).mean()
        etq = f"{sub_m['municipio'].iloc[0].split()[0].title()} ({len(sub_m)})".replace("Atizapan", "Atizapán")
        ax1.annotate(
            etq, xy=(cx, cy), xytext=(x_txt, y_txt),
            fontsize=8.5, fontweight="bold", color="#003366",
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#005a9c", alpha=0.92, lw=0.85),
            arrowprops=dict(arrowstyle="->", color="#005a9c", lw=1.0),
            path_effects=[pe.withStroke(linewidth=2, foreground="white")],
            zorder=8
        )

ax1.annotate(
    "Viento dominante\n(NE → SO)", xy=(510, 2180), xytext=(517.5, 2190),
    ha="center", fontsize=8, fontweight="bold", color="#7f0000",
    arrowprops=dict(arrowstyle="->", lw=2.0, color="#7f0000"), zorder=8
)

cbar = plt.colorbar(cf, ax=ax1, fraction=0.038, pad=0.025)
cbar.set_label(r"$\text{PM}_{2.5}$ industrial media ($\mu\text{g/m}^3$)", fontsize=8.5)

ax1.set_xlim(460, 524)
ax1.set_ylim(2112, 2194)
ax1.set_aspect("equal", adjustable="box")
ax1.set_xlabel("Coordenada Este UTM 14N (km)", fontsize=9.5)
ax1.set_ylabel("Coordenada Norte UTM 14N (km)", fontsize=9.5)
ax1.set_title("A. Fuentes pesadas e inmuebles hipervulnerables en la ZMVM", fontsize=10.5, fontweight="bold", pad=10)
ax1.grid(True, linestyle=":", alpha=0.4, color="#bdc3c7")
ax1.legend(loc="lower left", fontsize=7.6, framealpha=0.95, edgecolor="#bdc3c7")

# --- PANEL B: CONCENTRACIÓN MUNICIPAL Y BRECHA EDUCATIVA ---
ax2 = fig.add_subplot(gs[1])
conteo_mun = df_hiper["municipio"].value_counts().sort_values(ascending=True)
etiquetas_limpias = [
    m.title()
     .replace("De Juarez", "")
     .replace("De Morelos", "")
     .replace("De Baz", "")
     .replace("De Zaragoza", "")
     .replace("Tlahuac", "Tláhuac")
     .replace("Atizapan", "Atizapán")
     .strip()
    for m in conteo_mun.index
]

bars = ax2.barh(etiquetas_limpias, conteo_mun.values, color="#005a9c", edgecolor="#003366", height=0.56)
for bar, val in zip(bars, conteo_mun.values):
    pct = 100.0 * val / len(df_hiper)
    ax2.text(val + 0.3, bar.get_y() + bar.get_height()/2, f"{val} ({pct:.1f}%)", va="center", fontsize=8.8, fontweight="bold", color="#1a1a1a")

ax2.set_xlim(0, conteo_mun.max() + 4.5)
ax2.set_xlabel("Número de inmuebles escolares hipervulnerables (RECID)", fontsize=9.5)
ax2.set_title("B. Distribución municipal y métricas del grupo objetivo", fontsize=10.5, fontweight="bold", pad=10)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.grid(axis="x", linestyle=":", alpha=0.5)

texto_caja = (
    "Perfil del grupo objetivo (Decil 10 + Alta Marginación):\n"
    f"• Universo focalizado: {len(df_hiper)} inmuebles ({df_hiper['n_ccts'].sum()} CCT)\n"
    f"• Exposición media: {df_hiper['PM25_mean'].mean():.4f} μg/m³\n"
    f"• Aporte de fundiciones: {aporte_fund_hiper:.1f}% de la carga industrial\n"
    f"• Lenguaje (PLANEA): {df_hiper['lyc_mean'].mean():.1f} vs. {lyc_resto_pub:.1f} pts ({df_hiper['lyc_mean'].mean() - lyc_resto_pub:.1f} pts)\n"
    f"• Inversión HEPA ({n_aulas} aulas): ${capex_mxn:.2f}M MXN"
)
ax2.text(
    0.03, 0.05, texto_caja, transform=ax2.transAxes,
    fontsize=8.2, verticalalignment="bottom",
    bbox=dict(boxstyle="round,pad=0.5", facecolor="#f4f8fb", edgecolor="#005a9c", lw=1.0)
)

out_fig = f"{BASE}/figuras_tesis/fig_politica_hipervulnerables_cap4.png"
plt.savefig(out_fig, bbox_inches="tight", dpi=300)
plt.show()

print(f"\n--- FILAS EXACTAS PARA APÉNDICE C ({len(df_hiper)} INMUEBLES ÚNICOS) ---")
for idx, r in df_hiper.iterrows():
    mun_clean = (
        r["municipio"].title()
        .replace("De ", "de ")
        .replace("Juarez", "Juárez")
        .replace("Tlahuac", "Tláhuac")
        .replace("Atizapan", "Atizapán")
    )
    print(f"{idx+1} & \\texttt{{{r['RECID']}}} & \\texttt{{{r['ccts']}}} & {mun_clean} & {r['PM25_mean']:.4f} & {r['lyc_mean']:.1f} \\\\")


NameError: name 'BASE' is not defined